In [15]:
import sys
sys.path.append('..')

from utils.spark_session import get_spark_session
from utils.parquet_io import read_parquet, write_parquet
from pyspark.sql.functions import col, dayofweek, hour, when, round

# Initialize Spark session
spark = get_spark_session()
print('session created')

session created


In [16]:
# Read parquet files
taxi01_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\02_clean_data\\taxi01")
taxi02_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\02_clean_data\\taxi02")
zone_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\02_clean_data\\zone")

In [17]:
taxi01_df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2017-01-01 01:32:05|  2017-01-01 01:37:48|              1|          1.2|         1|                 N|         140|         236|           2|        6.5|  0.5|    0.5|       0.

In [18]:
# Add trip_duration_minutes column using timestampdiff
taxi01_df = taxi01_df.withColumn(
    "trip_duration_minutes",
    round(
        (col("tpep_dropoff_datetime").cast("long") - col("tpep_pickup_datetime").cast("long")) / 60, 2
    )
)

taxi02_df = taxi02_df.withColumn(
    "trip_duration_minutes",
    round(
        (col("tpep_dropoff_datetime").cast("long") - col("tpep_pickup_datetime").cast("long")) / 60, 2
    )
)


In [19]:
# Add average_speed_kmph column (trip distance / trip duration in hours with zero-division protection)
taxi01_df = taxi01_df.withColumn("average_speed_kmph", 
                                  when(col("trip_duration_minutes") > 0, 
                                       round(col("trip_distance") * 60 / col("trip_duration_minutes"), 2))
                                  .otherwise(0))

taxi02_df = taxi02_df.withColumn("average_speed_kmph", 
                                  when(col("trip_duration_minutes") > 0, 
                                       round(col("trip_distance") * 60 / col("trip_duration_minutes"), 2))
                                  .otherwise(0))

In [20]:
# Add is_weekend column (1 = Sunday, 7 = Saturday in Spark)
taxi01_df = taxi01_df.withColumn("is_weekend", 
                                  when((dayofweek(col("tpep_pickup_datetime")) == 1) | 
                                       (dayofweek(col("tpep_pickup_datetime")) == 7), 1)
                                  .otherwise(0))

taxi02_df = taxi02_df.withColumn("is_weekend", 
                                  when((dayofweek(col("tpep_pickup_datetime")) == 1) | 
                                       (dayofweek(col("tpep_pickup_datetime")) == 7), 1)
                                  .otherwise(0))


In [21]:
# Add is_rush_hour column (11 am to 7 pm on weekdays only)
taxi01_df = taxi01_df.withColumn("is_rush_hour", 
                                  when((hour(col("tpep_pickup_datetime")) >= 11) & 
                                       (hour(col("tpep_pickup_datetime")) < 19) &
                                       (col("is_weekend") == 0), 1)
                                  .otherwise(0))

taxi02_df = taxi02_df.withColumn("is_rush_hour", 
                                  when((hour(col("tpep_pickup_datetime")) >= 11) & 
                                       (hour(col("tpep_pickup_datetime")) < 19) &
                                       (col("is_weekend") == 0), 1)
                                  .otherwise(0))


In [22]:
# Show all new feature columns
taxi01_df.show(4)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------------------+------------------+----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|trip_duration_minutes|average_speed_kmph|is_weekend|is_rush_hour|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------------------+------------------+----------+------

In [23]:
# Show all new feature columns
taxi02_df.show(4)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------------------+------------------+----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|trip_duration_minutes|average_speed_kmph|is_weekend|is_rush_hour|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------------------+------------------+----------+------

In [24]:
# Save feature engineered data
write_parquet(taxi01_df, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\03_feature_data\\taxi01")
write_parquet(taxi02_df, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\03_feature_data\\taxi02")
write_parquet(zone_df, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\03_feature_data\\zone")
print("Feature engineered data saved successfully!")

Feature engineered data saved successfully!


In [25]:
spark.stop()